[来源：AgentScope官方Tutorial - Personal Research Assistant](https://docs.agentscope.io/tutorial/tutorial_research_agent)

## 第三方库导入及模型定义

In [1]:
import asyncio
import json
import os
import subprocess
import urllib.request
from datetime import datetime

from agentscope.agent import ReActAgent
from agentscope.formatter import OllamaChatFormatter
from agentscope.memory import InMemoryMemory,Mem0LongTermMemory
from agentscope.message import Msg, TextBlock
from agentscope.model import OllamaChatModel
from agentscope.embedding import OllamaTextEmbedding
from agentscope.tool import Toolkit, ToolResponse, execute_python_code

# import sys
# sys.path.append("../")
# from utils import print_markdown

api_key = os.environ.get("DASHSCOPE_API_KEY", "")

In [2]:
def download_file(url: str, path: str) -> None:
    """Download a file, falling back to curl if urllib fails."""
    try:
        urllib.request.urlretrieve(url, path)
    except Exception:
        # (Bad file descriptor). Use curl as a robust fallback.
        subprocess.run(
            ["curl", "-fSL", "-o", path, url],
            check=True,
        )


# ------------------------------------------------------------------ #
# Pre-download: fetch the paper PDF before any asyncio.run() calls,  #
# because repeated event-loop creation can corrupt urllib's sockets.  #
# ------------------------------------------------------------------ #
paper_url = "https://arxiv.org/pdf/1706.03762"
paper_path = os.path.join(
    os.getcwd(),
    "attention_is_all_you_need.pdf",
)

if not os.path.exists(paper_path):
    print("Downloading paper PDF (required for Part 4)...")
    download_file(paper_url, paper_path)
    print(f"Download complete: {paper_path}")
else:
    print(f"Paper already exists: {paper_path}")

Paper already exists: /Users/kang/Desktop/learning/JobSeeker/AgentScope/attention_is_all_you_need.pdf


OllamaChatModel替换成OllamaChatModel,本地运行模型

In [3]:
model = OllamaChatModel(
    model_name="qwen3.5:35b",
    stream=True,
)

assistant = ReActAgent(
    name="ResearchAssistant",
    sys_prompt=(
        "你是一名专业的研究助理。用户对人工智能开发尤为关注，尤其是大语言模型的研究进展。你的回答应当准确且条理清晰。"
    ),
    model=model,
    formatter=OllamaChatFormatter(),
    memory=InMemoryMemory(),
    toolkit=Toolkit(),

)

## 对话示例

jupyter notebook本身就是一个运行的时间循环，所以不能再次调用asyncio.run(demo_basic_chat())
，否则会报错。直接用await demo_basic_chat()

In [4]:
async def demo_basic_chat() -> None:
    """Basic conversation: the agent responds to the user's introduction."""
    response = await assistant(
        Msg(
            name = "user",
            content = "你好！我是一名计算机工程师，近来对大语言模型技术产生了浓厚兴趣。",
            role = "user",
        ),
    )
    print("Assistant:", response.get_text_content())


await demo_basic_chat()


ResearchAssistant(thinking): Thinking Process:

1.  **Analyze the Request:**
    *   **User Role:** Computer Engineer (计算机工程师).
    *   **User Interest:** Large Language Models (LLMs) / 大语言模型技术.
    *   **My Role:** Professional Research Assistant (专业的研究助理).
    *   **My Task:** Provide accurate and well-organized information about LLM research progress, tailored to a computer engineer's background.
    *   **Language:** Chinese.

2.  **Determine the Goal:**
    *   Acknowledge the user's background and interest.
    *   Provide a structured overview of the current state of LLM technology.
    *   Highlight key areas of research and development relevant to an engineer (architecture, training, applications, challenges, etc.).
    *   Maintain a professional, accurate, and clear tone.

3.  **Drafting the Content:**
    *   **Greeting:** Welcome the user, acknowledge their background.
    *   **Structure:** Break down the information into logical sections.
        *   Introduction/Overvie

没有定义传入模型行为控制参数的入口。

In [5]:
async def demo_multi_turn() -> None:
    """Multi-turn conversation: the agent remembers context."""
    await assistant(
        Msg(
            "user",
            "我目前在从事自然语言处理（NLP）方面的工作，主要关注Transformer架构的最新进展",
            "user",
        ),
    )
    response = await assistant(
        Msg("user", "你还记得我关注的是哪个技术领域吗？",
            "user"),
    )
    print("Assistant:", response.get_text_content())


await demo_multi_turn()

2026-04-10 23:42:39,487 | WARNING | _ollama_formatter:_format:235 - Unsupported block type thinking in the message, skipped.


ResearchAssistant(thinking): Thinking Process:

1.  **Analyze the User:**
    *   **Role:** Computer Engineer (now working in NLP).
    *   **Interest:** Large Language Models (LLMs), specifically Transformer architecture latest developments.
    *   **Goal:** Wants information on the latest progress in Transformer architecture for NLP work.
    *   **Tone:** Professional, accurate, structured, suitable for a research assistant.

2.  **Identify Key Topics:**
    *   Since the user is focused on *Transformer architecture* specifically (not just general LLM usage), I need to dig deeper into architectural innovations within or beyond the standard Transformer.
    *   Key areas to cover:
        *   Attention Mechanisms (Efficiency & Variants).
        *   Positional Encodings (Long context).
        *   Non-Transformer Alternatives (State Space Models, etc., since they challenge Transformer).
        *   Structural Optimizations (Mixture of Experts, Activation Functions).
        *   Rece

2026-04-10 23:44:21,724 | WARNING | _ollama_formatter:_format:235 - Unsupported block type thinking in the message, skipped.
2026-04-10 23:44:21,724 | WARNING | _ollama_formatter:_format:235 - Unsupported block type thinking in the message, skipped.


技术选型建议。
ResearchAssistant(thinking): Thinking Process:

1.  **Analyze the Request**:
    *   User asks: "你还记得我关注的是哪个技术领域吗？" (Do you remember which technical field I'm interested in?)
    *   Context: The user previously stated they are a computer engineer interested in LLM technology, specifically Transformer architecture advancements in NLP.
    *   Role: I am a professional research assistant.
    *   Goal: Accurately recall and confirm the user's stated interests based on the conversation history.

2.  **Review Conversation History**:
    *   User's first message: "你好！我是一名计算机工程师，近来对大语言模型技术产生了浓厚兴趣。" (Hello! I am a computer engineer, recently developed a strong interest in LLM technology.)
    *   My first response: Acknowledged their background, provided an overview of LLM tech (architecture, training, efficiency, applications).
    *   User's second message: "我目前在从事自然语言处理（NLP）方面的工作，主要关注 Transformer 架构的最新进展" (I am currently working in NLP, mainly focusing on the latest advancements i

## Tool Call 示例

In [6]:
def search_papers(query: str, limit: int = 5) -> ToolResponse:
    """Search for academic papers.

    Args:
        query (str):
            Search keywords, e.g. "transformer attention mechanism"
        limit (int):
            Number of results to return, default is 5
    """
    mock_papers = [
        {
            "title": "Attention Is All You Need",
            "authors": "Vaswani et al.",
            "year": 2017,
            "citations": 120000,
            "summary": "Proposes the Transformer architecture, replacing "
            "RNN/CNN with Self-Attention",
        },
        {
            "title": "BERT: Pre-training of Deep Bidirectional Transformers",
            "authors": "Devlin et al.",
            "year": 2018,
            "citations": 95000,
            "summary": "A pre-trained language model based on the "
            "Transformer Encoder",
        },
        {
            "title": "Language Models are Few-Shot Learners (GPT-3)",
            "authors": "Brown et al.",
            "year": 2020,
            "citations": 35000,
            "summary": "Demonstrates the few-shot learning capabilities "
            "of large-scale language models",
        },
    ]

    result = (
        f"Found {len(mock_papers)} papers related to \"{query}\":\n\n"
    )
    for i, p in enumerate(mock_papers[:limit], 1):
        result += (
            f"{i}. {p['title']} ({p['authors']}, {p['year']})\n"
            f"   Citations: {p['citations']} | Summary: {p['summary']}\n\n"
        )
    return ToolResponse(content=[TextBlock(type="text", text=result)])


def take_notes(title: str, content: str) -> ToolResponse:
    """Save research notes to a local file.

    Args:
        title (str):
            Note title
        content (str):
            Note content
    """
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
    with open("research_notes.md", "a", encoding="utf-8") as f:
        f.write(f"## [{timestamp}] {title}\n\n{content}\n\n---\n\n")
    return ToolResponse(
        content=[TextBlock(type="text", text=f"Note saved: {title}")],
    )

In [7]:
toolkit = Toolkit()
toolkit.register_tool_function(search_papers)
toolkit.register_tool_function(execute_python_code)
toolkit.register_tool_function(take_notes)

print("Registered tools:")
for name in toolkit.tools:
    print(f"  - {name}")

print("\nTool JSON Schema:")
print(json.dumps(toolkit.get_json_schemas(), indent=2, ensure_ascii=False))

Registered tools:
  - search_papers
  - execute_python_code
  - take_notes

Tool JSON Schema:
[
  {
    "type": "function",
    "function": {
      "name": "search_papers",
      "parameters": {
        "properties": {
          "query": {
            "description": "Search keywords, e.g. \"transformer attention mechanism\"",
            "type": "string"
          },
          "limit": {
            "default": 5,
            "description": "Number of results to return, default is 5",
            "type": "integer"
          }
        },
        "required": [
          "query"
        ],
        "type": "object"
      },
      "description": "Search for academic papers."
    }
  },
  {
    "type": "function",
    "function": {
      "name": "execute_python_code",
      "parameters": {
        "properties": {
          "code": {
            "description": "The Python code to be executed.",
            "type": "string"
          },
          "timeout": {
            "default": 300,
       

In [8]:
type(toolkit.get_json_schemas()[0])

dict

In [9]:
research_agent = ReActAgent(
    name="ResearchAssistant",
    sys_prompt=(
        "你是一个服务于对AI和Transformer技术感兴趣的计算机工程师的研究助手。"
        "当用户的问题需要搜索论文或进行计算时，主动使用适当的工具。"
        "使用take_notes工具记录重要信息。"
    ),
    model=model,
    formatter=OllamaChatFormatter(),
    memory=InMemoryMemory(),
    toolkit=toolkit,
)


async def demo_mock_tools() -> None:
    """Demonstrate the agent with mock tools."""
    # The agent will automatically decide to call search_papers
    await research_agent(
        Msg("user", "帮我搜索关于 Transformer "
            "architecture", "user"),
    )

    # The agent will call execute_python_code
    await research_agent(
        Msg(
            "user",
            "请计算 BERT-base: "
            "12 layers * 768 dims * 768 dims * 4, and return the result",
            "user",
        ),
    )

    # The agent will call take_notes
    await research_agent(
        Msg(
            "user",
            "请记录下 Transformer Self-Attention mechanism, 使处理顺序数据成为可能",
            "user",
        ),
    )


await demo_mock_tools()

ResearchAssistant(thinking): 用户想要搜索关于Transformer architecture的论文。这是一个关于AI和深度学习架构的查询，我应该使用search_papers工具来搜索相关的学术论文。

我将使用一个清晰的查询词来搜索Transformer架构相关的论文。

2026-04-10 23:45:13,123 | WARNING | _ollama_formatter:_format:235 - Unsupported block type thinking in the message, skipped.



{
    "type": "tool_use",
    "id": "0_search_papers",
    "name": "search_papers",
    "input": {
        "query": "Transformer architecture",
        "limit": 5
    }
}

system: {
    "type": "tool_result",
    "id": "0_search_papers",
    "name": "search_papers",
    "output": [
        {
            "type": "text",
            "text": "Found 3 papers related to \"Transformer architecture\":\n\n1. Attention Is All You Need (Vaswani et al., 2017)\n   Citations: 120000 | Summary: Proposes the Transformer architecture, replacing RNN/CNN with Self-Attention\n\n2. BERT: Pre-training of Deep Bidirectional Transformers (Devlin et al., 2018)\n   Citations: 95000 | Summary: A pre-trained language model based on the Transformer Encoder\n\n3. Language Models are Few-Shot Learners (GPT-3) (Brown et al., 2020)\n   Citations: 35000 | Summary: Demonstrates the few-shot learning capabilities of large-scale language models\n\n"
        }
    ]
}


2026-04-10 23:45:22,433 | WARNING | _ollama_formatter:_format:235 - Unsupported block type thinking in the message, skipped.


ResearchAssistant: {
    "type": "tool_use",
    "id": "0_take_notes",
    "name": "take_notes",
    "input": {
        "title": "Transformer Architecture Overview",
        "content": "Key papers on Transformer Architecture:\n1. \"Attention Is All You Need\" (Vaswani et al., 2017) - The foundational paper introducing the Transformer model, which replaced RNNs and CNNs with a Self-Attention mechanism. Over 120,000 citations.\n2. \"BERT: Pre-training of Deep Bidirectional Transformers\" (Devlin et al., 2018) - Introduced BERT, a language model using the Transformer Encoder for deep bidirectional representations. ~95,000 citations.\n3. \"Language Models are Few-Shot Learners\" (GPT-3) (Brown et al., 2020) - Showcased the power of large-scale Transformers (GPT-3) in few-shot and zero-shot learning. ~35,000 citations.\n\nCore concept: The Transformer relies on self-attention mechanisms (Scaled Dot-Product Attention) to process sequences in parallel, allowing for better handling of long-ter

...


{
    "type": "tool_use",
    "id": "0_execute_python_code",
    "name": "execute_python_code",
    "input": {
        "code": "# 计算 BERT-base 参数量\nlayers = 12\nhidden_dim = 768\nnum_heads = 12  # BERT-base typical value\n\n# 按照用户提供的公式计算\nresult = layers * hidden_dim * hidden_dim * 4\nprint(f\"BERT-base estimated parameters: {result:,}\")\nprint(f\"\\nCalculation breakdown:\")\nprint(f\"{layers} layers * {hidden_dim} dims * {hidden_dim} dims * 4 = {result}\")\n\n# 更详细的BERT-base实际参数量说明\nprint(f\"\\n--- BERT-base 实际架构 ---\")\nprint(\"BERT-base has approximately 110 million parameters in total\")\nprint(\"The formula provided estimates self-attention and feed-forward layers only\")",
        "timeout": 10
    }
}

system: {
    "type": "tool_result",
    "id": "0_execute_python_code",
    "name": "execute_python_code",
    "output": [
        {
            "type": "text",
            "text": "<returncode>0</returncode><stdout>BERT-base estimated parameters: 28,311,552\n\nCalculation brea

2026-04-10 23:46:05,324 | WARNING | _ollama_formatter:_format:235 - Unsupported block type thinking in the message, skipped.
2026-04-10 23:46:05,324 | WARNING | _ollama_formatter:_format:235 - Unsupported block type thinking in the message, skipped.



ResearchAssistant(thinking): 用户要求我记录关于Transformer Self-Attention机制的信息，说明它如何使处理顺序数据成为可能。我应该使用take_notes工具来保存这个重要的技术概念。

我需要记录的内容应该包括：
1. Self-Attention的基本原理
2. 它是如何处理序列数据的
3. 为什么它比RNN/CNN更适合处理顺序数据
4. 关键机制（Query, Key, Value）

让我创建一个详细的笔记。

2026-04-10 23:46:31,829 | WARNING | _ollama_formatter:_format:235 - Unsupported block type thinking in the message, skipped.
2026-04-10 23:46:31,829 | WARNING | _ollama_formatter:_format:235 - Unsupported block type thinking in the message, skipped.
2026-04-10 23:46:31,829 | WARNING | _ollama_formatter:_format:235 - Unsupported block type thinking in the message, skipped.



{
    "type": "tool_use",
    "id": "0_take_notes",
    "name": "take_notes",
    "input": {
        "title": "Transformer Self-Attention Mechanism",
        "content": "# Transformer Self-Attention Mechanism\n\n## 核心概念\nSelf-Attention（自注意力机制）是Transformer架构的核心，它使模型能够**并行处理序列数据**，而不需要像RNN那样按顺序处理。\n\n## 工作机制\n\n### 1. QKV 矩阵\n每个输入 token 通过三个线性变换得到：\n- **Query (Q)**: 表示当前token想要查询什么\n- **Key (K)**: 表示当前token可以被什么查询\n- **Value (V)**: 包含当前token的实际内容\n\n### 2. 注意力分数计算\n```\nAttention(Q, K, V) = softmax(QK^T / sqrt(d_k)) * V\n```\n- **QK^T**: 计算每个token与其他token的相似度\n- **sqrt(d_k)**: 缩放因子，防止点积过大导致梯度消失\n- **softmax**: 归一化为概率分布\n- **V加权求和**: 根据注意力权重聚合信息\n\n### 3. Multi-Head Attention\n- 将隐藏维度分为多个\"头\"（BERT-base: 12 heads）\n- 每个头独立学习不同的注意力模式\n- 最终拼接所有头的输出\n\n## 处理顺序数据的优势\n\n### 对比RNN：\n| 特征 | RNN | Self-Attention |\n|------|-----|----------------|\n| 处理顺序 | 必须顺序处理 | 完全并行 |\n| 长依赖 | 梯度消失问题 | 全局感受野 |\n| 计算效率 | O(n) 序列依赖 | O(n²) 但可并行 |\n\n### 关键能力：\n1. **全局上下文感知**：任意位置可以直接访问序列中的所有信息\n2. **位置无关性**：通过

## MCP 示例

In [10]:

from agentscope.mcp import StdIOStatefulClient

async def demo_tavily_mcp():
    tavily_client = StdIOStatefulClient(
        name="tavily",
        command="npx",
        args=["-y", "tavily-mcp@latest"],
        env={"TAVILY_API_KEY": os.environ["TAVILY_API_KEY"]},
    )
    await tavily_client.connect()

    mcp_toolkit = Toolkit()
    await mcp_toolkit.register_mcp_client(tavily_client)
    mcp_toolkit.register_tool_function(execute_python_code)
    mcp_toolkit.register_tool_function(take_notes)

    mcp_agent = ReActAgent(
        name="ResearchAssistant",
        sys_prompt="你是一个专业的研究助手，使用搜索工具获取最新的信息。",
        model=model,
        formatter=OllamaChatFormatter(),
        memory=InMemoryMemory(),
        toolkit=mcp_toolkit,
    )

    await mcp_agent(
        Msg("user", "搜索最新 Transformer 架构改进方案，时间范围为 2026 年", "user"),
    )
    await tavily_client.close()

await demo_tavily_mcp()


2026-04-10 23:46:46,682 | INFO    | _stateful_client_base:connect:66 - MCP client connected.
2026-04-10 23:46:46,686 | INFO    | _toolkit:register_mcp_client:1169 - Registered 5 tool functions from MCP: tavily_search, tavily_extract, tavily_crawl, tavily_map, tavily_research.


ResearchAssistant(thinking): 用户要求搜索关于最新 Transformer 架构改进方案的信息，时间范围限定在2026年。这是一个比较具体的技术搜索请求。

我需要使用 tavily_search 工具来搜索这个主题。由于用户指定了时间范围是2026年，我需要设置相应的搜索参数。

让我进行一次搜索来查找2026年的Transformer架构改进方案相关信息。
{
    "type": "tool_use",
    "id": "0_tavily_search",
    "name": "tavily_search",
    "input": {
        "query": "Transformer架构改进方案 最新研究 2026",
        "search_depth": "advanced",
        "time_range": "year",
        "max_results": 10
    }
}



2026-04-10 23:47:00,038 | WARNING | _ollama_formatter:_format:235 - Unsupported block type thinking in the message, skipped.


system: {
    "type": "tool_result",
    "id": "0_tavily_search",
    "name": "tavily_search",
    "output": [
        {
            "type": "text",
            "text": "Detailed Results:\n\nTitle: 老牌Transformer杀手在ICLR悄然更新：Mamba-3三大改进趋近设计完全体|改进_新浪科技_新浪网\nURL: https://finance.sina.com.cn/tech/roll/2025-10-14/doc-inftwafi1991416.shtml\nContent: 这种更新规则在设计上仍保持训练和推理的轻量级特性，同时克服了当前许多线性模型在状态追踪能力上的不足。研究团队指出，这种复数更新机制等价于一种数据依赖的旋转位置编码，因此可以高效计算。\n\n多输入多输出状态空间模型（MIMO SSM）\n\n为了提升解码阶段的 FLOP 利用效率，研究团队将状态更新方式从基于外积（outer-product）的形式转换为基于矩阵乘法的形式。从 SSM 的信号处理基础来看，这一转变正对应于从单输入单输出（SISO）动态系统向多输入多输出（MIMO）动态系统的泛化。\n\nMamba-3 可以多通道同时更新状态，极大提升 GPU 并行吞吐效率。\n\nMIMO 形式尤其适合推理阶段，因为其额外的表达能力允许在状态更新中投入更多计算量，而无需增加状态大小，从而不影响速度。\n\n同时，研究团队也对整体架构进行调整，使其更贴近基线 Transformer 架构。Mamba-3 用更常见的 QK-normalization 替换了输出前投影归一化机制，并将短卷积设为可选项。\n\n对比 Mamba-2 与 Mamba-3 的架构升级\n\n实证验证\n\n研究团队在一系列合成任务和语言建模任务上对新模型进行实证验证：\n\n更好的质量（Better Quality）\n\n在标准下游语言建模评测中，Mamba-3 的表现达到或超过 Mamba-2 及其他开源架构。例如，Mamba-3-1.5B 在所有下游任务上的平均准确率优于其 Transformer、Mamba-

## 记忆功能

In [11]:
memory = InMemoryMemory()


async def demo_short_term_memory() -> None:
    """Short-term memory: add, retrieve, and check size."""
    await memory.add(
        Msg("user", "我是一名计算机工程师，主要从事智能体应用开发工作", "user"),
    )
    await memory.add(
        Msg("assistant", "Hello! 智能体应用开发涉及许多有趣的技术。", "assistant"),
    )
    await memory.add(
        Msg("user", "最近我 对大型语言模型和Transformer 架构感兴趣。", "user"),
    )
    await memory.add(
        Msg(
            "assistant",    
            "Transformer 架构是当今人工智能领域最重要的架构之一。",
            "assistant",
        ),
    )

    messages = await memory.get_memory()
    print(f"Memory contains {len(messages)} messages")
    size = await memory.size()
    print(f"Memory size: {size}")


await demo_short_term_memory()


Memory contains 4 messages
Memory size: 4


In [12]:
async def demo_memory_marks() -> None:
    """Memory Marks: categorize and filter messages."""
    mark_memory = InMemoryMemory()

    await mark_memory.add(
        Msg("User", "Search for Transformer papers", "user"),
    )
    await mark_memory.add(
        Msg(
            "system",
            "User is a computer engineer interested in AI and "
            "Transformer technologies",
            "system",
        ),
        marks="hint",
    )
    await mark_memory.add(
        Msg("system", "Called the search_papers tool", "system"),
        marks="tool",
    )

    hints = await mark_memory.get_memory(mark="hint")
    chat = await mark_memory.get_memory(exclude_mark="tool")
    print(f"hint messages: {len(hints)}, after excluding tool: {len(chat)}")

await demo_memory_marks()

hint messages: 1, after excluding tool: 2


In [13]:
from agentscope.token import CharTokenCounter

agent_with_compression = ReActAgent(
    name="ResearchAssistant",
    sys_prompt="You are a research assistant.",
    model=model,
    formatter=OllamaChatFormatter(),
    memory=InMemoryMemory(),
    toolkit=Toolkit(),
    compression_config=ReActAgent.CompressionConfig(
        enable=True,
        agent_token_counter=CharTokenCounter(),
        trigger_threshold=3000,
        keep_recent=5,
    ),
)

In [14]:
async def demo_persistence() -> None:
    """Memory persistence: export → save to file → restore."""
    persist_memory = InMemoryMemory()
    await persist_memory.add(
        Msg("user", "My research focus is NLP and Transformer", "user"),
    )

    state = persist_memory.state_dict()
    state_file = "/tmp/memory_state.json"
    with open(state_file, "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False)
    print(f"State saved to: {state_file}")

    new_memory = InMemoryMemory()
    with open(state_file, "r", encoding="utf-8") as f:
        new_memory.load_state_dict(json.load(f))
    
    restored = await new_memory.get_memory()
    print(f"Restored {len(restored)} messages")


await demo_persistence()


State saved to: /tmp/memory_state.json
Restored 1 messages


In [15]:
import os
from mem0.vector_stores.configs import VectorStoreConfig

vector_store_config = VectorStoreConfig(
    provider="qdrant",
    config={
        "path": f"{os.getcwd()}/qdrant_storage",  # Qdrant 数据存储路径
    }
)


In [16]:
long_term_memory = Mem0LongTermMemory(
    agent_name="ResearchAssistant",
    user_name="researcher",
    model=OllamaChatModel(
        model_name="qwen3.5:35b",
        stream=False,
    ),
    embedding_model=OllamaTextEmbedding(  
        model_name="qwen3-embedding:8b",
        dimensions=1536, 
    ),
    # vector_store_config=vector_store_config,
    # 连接到本地启动的 Qdrant 服务器，默认端口为 6333
    qdrant_url="http://localhost:6333",
    # 如果 Qdrant 服务器需要 API key，请取消下面注释并设置
    # qdrant_api_key="your-api-key",
    on_disk=False,  # 使用服务器模式，禁用本地磁盘存储
)

In [17]:
async def demo_long_term_memory():
    # Record user profile
    await long_term_memory.record_to_memory(
        thinking="Recording user's background and research interests",
        content=[
            "User is a computer science engineer working on LLM Agent development",
            "User is very interested in large language models and the Transformer architecture",
            "User is studying the Attention Is All You Need paper",
            "User prefers using Python for technical experiments",
        ],
    )

    # Retrieve
    result = await long_term_memory.retrieve_from_memory(
        keywords=["research interests", "technical background"],
        limit=1,
    )
    print(result)
    for block in result.content:
        print(block["text"],"hehe")

await demo_long_term_memory()

ToolResponse(content=[{'type': 'text', 'text': "Recording user's background and research interests\nUser is a computer science engineer working on LLM Agent development\nUser is very interested in large language models and the Transformer architecture\nUser is studying the Attention Is All You Need paper\nUser prefers using Python for technical experiments\nRecording user's background and research interests\nUser is a computer science engineer working on LLM Agent development\nUser is very interested in large language models and the Transformer architecture\nUser is studying the Attention Is All You Need paper\nUser prefers using Python for technical experiments"}], metadata=None, stream=False, is_last=True, is_interrupted=False, id='2026-04-10 23:53:01.006_29a7cb')
Recording user's background and research interests
User is a computer science engineer working on LLM Agent development
User is very interested in large language models and the Transformer architecture
User is studying the 

In [18]:
agent = ReActAgent(
    name="ResearchAssistant",
    sys_prompt="You are a research assistant with long-term memory "
               "capabilities. ...",
    model=model,
    formatter=OllamaChatFormatter(),
    memory=InMemoryMemory(),
    toolkit=toolkit,
    long_term_memory=long_term_memory,
    long_term_memory_mode="agent_control",
)

## RAG

In [19]:
print(f"Using paper: {paper_path}")

Using paper: /Users/kang/Desktop/learning/JobSeeker/AgentScope/attention_is_all_you_need.pdf


In [20]:
from agentscope.embedding import OllamaTextEmbedding
from agentscope.rag import PDFReader, SimpleKnowledge, QdrantStore

embedding_model = OllamaTextEmbedding(
    model_name="qwen3-embedding:8b",
    dimensions=1024,
)


async def build_knowledge() -> SimpleKnowledge:
    """Download paper → parse with PDFReader → write to vector store."""
    vector_store = QdrantStore(
        location=":memory:",
        collection_name="transformer_paper",
        dimensions=1024,
    )
    knowledge = SimpleKnowledge(
        embedding_model=embedding_model,
        embedding_store=vector_store,
    )

    pdf_reader = PDFReader(chunk_size=1024, split_by="paragraph")
    docs = await pdf_reader(pdf_path=paper_path)
    print(f"Paper split into {len(docs)} document chunks")

    await knowledge.add_documents(docs)
    print("Knowledge base construction complete")
    return knowledge


knowledge = await build_knowledge()


2026-04-10 23:53:01,504 | INFO    | _text_reader:__call__:68 - Reading text with chunk_size=1024, split_by=paragraph
2026-04-10 23:53:01,505 | INFO    | _text_reader:__call__:126 - Finished splitting the text into 847 chunks.


Paper split into 847 document chunks
Knowledge base construction complete


In [21]:
async def demo_agentic_rag() -> None:
    """Agentic RAG: the agent uses the retrieve_knowledge tool."""
    rag_toolkit = Toolkit()
    rag_toolkit.register_tool_function(search_papers)
    rag_toolkit.register_tool_function(execute_python_code)
    rag_toolkit.register_tool_function(
        knowledge.retrieve_knowledge,
        func_description=(
            "Retrieve relevant information from the Transformer paper "
            "knowledge base. Use this when the user asks about technical "
            "details such as Transformer architecture, Self-Attention, "
            "Multi-Head Attention, etc."
        ),
    )

    rag_agent = ReActAgent(
        name="ResearchAssistant",
        sys_prompt=(
            "You are a research assistant serving a computer engineer "
            "interested in AI. "
            "You have a knowledge base built from the Transformer paper. "
            "When the user asks about technical details, first use "
            "retrieve_knowledge to search the paper content, "
            "then answer based on the retrieved results. "
            "If the knowledge base does not contain relevant information, "
            "honestly state so."
        ),
        model=model,
        formatter=OllamaChatFormatter(),
        memory=InMemoryMemory(),
        toolkit=rag_toolkit,
    )

    await rag_agent(
        Msg(
            "User",
            "How does Multi-Head Attention work in Transformer?",
            "user",
        ),
    )
    await rag_agent(
        Msg(
            "User",
            "What method does the paper use for Positional Encoding?",
            "user",
        ),
    )


await demo_agentic_rag()


ResearchAssistant(thinking): The user is asking about Multi-Head Attention in the Transformer architecture. This is a technical question about the Transformer paper, so I should use the retrieve_knowledge function to search for relevant information in the Transformer paper knowledge base.

I need to craft a specific and concise query that will help retrieve information about Multi-Head Attention. Let me search for this topic.
{
    "type": "tool_use",
    "id": "0_retrieve_knowledge",
    "name": "retrieve_knowledge",
    "input": {
        "query": "Multi-Head Attention mechanism",
        "limit": 5,
        "score_threshold": "0.7"
    }
}



2026-04-11 00:03:22,677 | WARNING | _ollama_formatter:_format:235 - Unsupported block type thinking in the message, skipped.


system: {
    "type": "tool_result",
    "id": "0_retrieve_knowledge",
    "name": "retrieve_knowledge",
    "output": [
        {
            "type": "text",
            "text": "Error: ufunc 'less' did not contain a loop with signature matching types (<class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.StrDType'>) -> None"
        }
    ]
}
ResearchAssistant: {
    "type": "tool_use",
    "id": "0_retrieve_knowledge",
    "name": "retrieve_knowledge",
    "input": {
        "query": "Multi-Head Attention",
        "limit": 5
    }
}


2026-04-11 00:03:30,998 | WARNING | _ollama_formatter:_format:235 - Unsupported block type thinking in the message, skipped.


system: {
    "type": "tool_result",
    "id": "0_retrieve_knowledge",
    "name": "retrieve_knowledge",
    "output": [
        {
            "type": "text",
            "text": "Score: 0.9970124651270371, Content:  Multi-Head Attention"
        },
        {
            "type": "text",
            "text": "Score: 0.9537825409068936, Content: 3.2.2 Multi-Head Attention"
        },
        {
            "type": "text",
            "text": "Score: 0.9479128094590086, Content: multi-headed self-attention."
        },
        {
            "type": "text",
            "text": "Score: 0.9224048077035983, Content: Scaled Dot-Product Attention"
        },
        {
            "type": "text",
            "text": "Score: 0.8792038472923869, Content: Multi-head attention allows the model to jointly attend to information from different representation"
        }
    ]
}
ResearchAssistant: Based on the Transformer paper, **Multi-Head Attention** works by allowing the model to jointly attend to info

2026-04-11 00:03:49,430 | WARNING | _ollama_formatter:_format:235 - Unsupported block type thinking in the message, skipped.


 input data.
ResearchAssistant(thinking): The user is asking about the positional encoding method used in the Transformer paper. I should use the retrieve_knowledge function to search for information about positional encoding in the knowledge base built from the Transformer paper.
{
    "type": "tool_use",
    "id": "0_retrieve_knowledge",
    "name": "retrieve_knowledge",
    "input": {
        "query": "positional encoding",
        "limit": 5
    }
}



2026-04-11 00:03:54,789 | WARNING | _ollama_formatter:_format:235 - Unsupported block type thinking in the message, skipped.
2026-04-11 00:03:54,789 | WARNING | _ollama_formatter:_format:235 - Unsupported block type thinking in the message, skipped.


system: {
    "type": "tool_result",
    "id": "0_retrieve_knowledge",
    "name": "retrieve_knowledge",
    "output": [
        {
            "type": "text",
            "text": "Score: 0.9144925981342441, Content: 3.5 Positional Encoding"
        },
        {
            "type": "text",
            "text": "Score: 0.8298284844964856, Content: bottoms of the encoder and decoder stacks. The positional encodings have the same dimension dmodel"
        },
        {
            "type": "text",
            "text": "Score: 0.8168720372172342, Content: P E(pos,2i+1) = cos(pos/100002i/dmodel )"
        },
        {
            "type": "text",
            "text": "Score: 0.8019830296010964, Content: tokens in the sequence. To this end, we add \"positional encodings\" to the input embeddings at the"
        },
        {
            "type": "text",
            "text": "Score: 0.7949157671337239, Content: P E(pos,2i) = sin(pos/100002i/dmodel )"
        }
    ]
}
ResearchAssistant: According to th

In [23]:
import agentscope

In [24]:
agentscope.__version__

'1.0.18'

In [25]:
agentscope.init()


In [26]:
?agentscope.init

Signature:
agentscope.init(
    project: str | None = None,
    name: str | None = None,
    run_id: str | None = None,
    logging_path: str | None = None,
    logging_level: str = 'INFO',
    studio_url: str | None = None,
    tracing_url: str | None = None,
) -> None
Docstring:
Initialize the agentscope library.

Args:
    project (`str | None`, optional):
        The project name.
    name (`str | None`, optional):
        The name of the run.
    run_id (`str | None`, optional):
        The identity of a running instance, which can be an agent, or a
        multi-agent system. The `run_id` is used in AgentScope-Studio to
        distinguish different runs.
    logging_path (`str | None`, optional):
        The path to saving the log file. If not provided, logs will not be
        saved.
    logging_level (`str | None`, optional):
        The logging level. Defaults to "INFO".
    studio_url (`str | None`, optional):
        The URL of the AgentScope Studio to connect to.
    traci

In [27]:
?model

Signature:     
model(
    messages: list[dict[str, typing.Any]],
    tools: list[dict] | None = None,
    tool_choice: Union[Literal['auto', 'none', 'required'], str, NoneType] = None,
    structured_model: Optional[Type[pydantic.main.BaseModel]] = None,
    **kwargs: Any,
) -> Union[agentscope.model._model_response.ChatResponse, AsyncGenerator[agentscope.model._model_response.ChatResponse, NoneType]]
Type:           OllamaChatModel
String form:    <agentscope.model._ollama_model.OllamaChatModel object at 0x12151a3c0>
File:           ~/miniforge3/lib/python3.13/site-packages/agentscope/model/_ollama_model.py
Docstring:      The Ollama chat model class in agentscope.
Init docstring:
Initialize the Ollama chat model.

Args:
    model_name (`str`):
        The name of the model.
    stream (`bool`, default `True`):
        Streaming mode or not.
    options (`dict`, default `None`):
        Additional parameters to pass to the Ollama API. These can
        include temperature etc.
    ke